# Running the analyses on Colab

This notebook runs every analysis in the repo on a Colab GPU. The data is downloaded to the runtime's local disk, and the results and figures are saved to Google Drive, so a disconnect only costs the subject that was running.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DChristensen12/DeepBrain--Brain-alignment-AI/blob/main/notebooks/colab.ipynb)

## Setup

First, make sure the runtime has a GPU (`Runtime` → `Change runtime type`). The encoding models run on CPU too, just much slower.

In [ ]:
!nvidia-smi

Next, we mount Google Drive, clone the repo, and install the packages Colab doesn't already have.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
![ -d /content/deepbrain ] && git -C /content/deepbrain pull || git clone https://github.com/DChristensen12/DeepBrain--Brain-alignment-AI.git /content/deepbrain
%cd /content/deepbrain
!pip install -q -r requirements.txt

We link the `results` and `figures` folders to Drive. Everything the scripts and notebooks write ends up there, and it survives the runtime being reset.

In [ ]:
drive_dir = "/content/drive/MyDrive/deepbrain"  # change this to save somewhere else

!mkdir -p {drive_dir}/results {drive_dir}/figures
!rm -rf results figures
!ln -s {drive_dir}/results results
!ln -s {drive_dir}/figures figures

## Download the data

This downloads the preprocessed high-gamma data for all 9 subjects, the podcast audio and transcript, and the five feature spaces from OpenNeuro (about 6.4 GB). Files that already exist are skipped.

In [ ]:
!make download

## Check one subject first

Before running everything, we fit the GPT-2 XL encoding model for sub-03 only. This is the subject the tutorials use, so the brain map below should look like the one at the end of the encoding tutorial, with more electrodes since we keep all of them and not just the grids.

In [ ]:
!python code/encoding.py --modelname=gpt2-xl --layer=24 --subject=03

In [ ]:
import h5py
import matplotlib.pyplot as plt
from nilearn.plotting import plot_markers

with h5py.File("results/sub-03/encoding/sub-03_task-podcast_desc-gpt2.xl.l24_result.h5", "r") as f:
    scores = f["scores"][...].mean(0)
    coords = f["coords"][...]
print(f"Scores have a shape of: {scores.shape}")

values = scores.max(-1)
order = values.argsort()
plot_markers(values[order], coords[order],
             node_size=30, display_mode="lzr",
             node_vmin=0, node_cmap="inferno_r", colorbar=True)
plt.show()

## Quality checks

The cross-correlation and ERP analyses only use the CPU and take a few minutes.

In [ ]:
!make audioxcorr erp

## Encoding models

This fits all five feature spaces for all 9 subjects. Each finished subject is saved right away and skipped on the next run, so if the runtime disconnects, rerun the setup cells and then this one.

Note: this is the long step. The paper's code budgeted about 70 minutes per feature space on a GPU.

In [ ]:
!make encoding

## Figures

Finally, we execute the two figure notebooks. The SVGs are written to `figures/` on Drive.

In [ ]:
!make figures